# A no-nonsense guide to convergence-entropy and convergence-information

In [ ]:
import torch.nn as nn
import torch
import numpy as np
import plotly.graph_objects as go

## Playing with guts of large language models

If you want to know all the different kinds of (large) language models out there and what each one is "optimized" to do, check out [https://huggingface.co/](https://huggingface.co/)

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig

In [ ]:
# name of the specific LLM we're going to use/dissect
llm_name = 'roberta-base'

First things we're going to do is set up a specialized "config" class. a config tells the class what it should behave like... this is an advanced python skill, and not all classes require or use them. In fact, most don't.

In [ ]:
config = AutoConfig.from_pretrained(llm_name, output_hidden_states=True)

That last part in the "function" call for the class is important. That'll return all of the guts of the model to us every time we pass a string to it so we can see everything that changed inside of it when processing the text.

The next part is also important. LLMs don't actually read the text directly. This is true in two ways. First, the LLM never sees the linguistic strings themselves, they're converted to numerical tokens (it's a dictionary that maps a number to a word, or a part of a word in a lot of cases). This is what gets passed to the LLM as an input. Tokenizers handle that dicitonary part.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(llm_name)

Last bit of setup here. We need to actually set up the LLM's class. I'm going to name it mody. Because "mody" is cute.

In [ ]:
mod = AutoModel.from_pretrained(llm_name, config=config)

Notice that the class-function pair we're using here says "pretrained"? That's because this model, before we downloaded it to used, was already trained to predict tokens in texts on a huge corpus of internet data. That training is important. If you don't train a model, it just generated random outputs. So we're using a class written be people at hugging face, to download and use a large-language model that was trained by someone else (usually taking a lot of time, effort, and resources). It's worth just remarking on that for a moment and remembering our work is just the tip of a long line of people doing a ton of work (so cite them when you can!!!!!)

### Understanding tokenization

Before we go too much further, let's quickly get acquainted with the tokenizer. This is easy. We just want to pass it a text, see what comes out, and then convert it back to something that's mostly human readable.

In [ ]:
a_sentence = 'I was really disappointed this morning that I was out of croissants'

converting it to tokens in 3... 2.... 1...

In [ ]:
token_ids = tokenizer.encode(a_sentence)
token_ids

What do you see here? Gimme a description :)

And just so you get an intuition of why there are so many tokens in there, if we convert it back to strings...

In [ ]:
token_texts = tokenizer.convert_ids_to_tokens(token_ids)
token_texts

Remember this: tokens are not 1:1 equivalent to words!

### Generating embeddings/vectors

Okay. Recall how I said LLMs don't actually read texts and that's true on two levels? Here's what that means. When you pass an LLM a token(_id), that token is actually passed to a ginormous artificial neural network (ANN). For what it's worth, an ANN is basically the answer to the question "what if I stacked linear regression on top of linear regression?".

Every token you pass an ANN changes what the "model weights". The "guts" of the model. If you train the model to basically act as autocomplete, the guts of the model start to learn and internalize something that's similar to semantics. It's not actually semantics, but it's got a high correlation with it, and that makes it useful. So people often say that the guts of these models are "semantic representations" of the meaning of tokens, sentences, etc. Here's how you pull those "semantic representations" out of the LLM.

In [ ]:
results = mod(
  torch.LongTensor(token_ids).unsqueeze(0)
)

Some quick notes:
1. torch is basically a clone of numpy. But it's a clone that allows you to run things on a GPU, which makes computation much, much faster. We're not using a GPU right now, but we will.
2. `torch.LongTensor` is effectively the same thing as n`p.array(alist).astype(int)`
3. It's a dumb convention, but it's hard coded into the class. HuggingFace models (LLMs), require the inputs to have a 3D shape. That is: 1 X 1 X #\_of]_token\_ids. Our list of token ids is only 2D (1 x #\_of]_token\_ids). `.unsqueeze(0)` adds another dimension to the very beggining of our `token_ids` array.

Hence `torch.LongTensor(token_ids).unsqueeze(0)`

### Comparing utterances

In [ ]:
a_sentence = 'How many times do I have to tell you: you do not eat the tiny packet that you find in coat pockets in the department store.'
a_token_ids = tokenizer.encode(a_sentence)
a_token_strings = tokenizer.convert_ids_to_tokens(a_token_ids)
a_results = mod(torch.LongTensor(a_token_ids).unsqueeze(0))

In [ ]:
b_sentence = "But they taste like grandma's house!"
b_token_ids = tokenizer.encode(b_sentence)
b_token_strings = tokenizer.convert_ids_to_tokens(b_token_ids)
b_results = mod(torch.LongTensor(b_token_ids).unsqueeze(0))

In [ ]:
# before moving on, check out the shape of each of these things. Make sure you track the "dimensions" of each of the objects so that you have some intuitions behind what each part of the process is doing.

The thing of it is, as you might have noticed, that the .hidden_states has a weird structure. It's a tuple with a bunch of "tensors" that have shape 1 X #\_of\_tokens X 768. Each item in the tuple is actually one layer of the neural network. And different layers actually are better at retaining different kinds of information about things (layer 8 (7 in python speak) tends to be the best for semantics). See [[1]](https://aclanthology.org/D19-1006/), [[2]](http://arxiv.org/abs/2002.04815), [[3]](https://aclanthology.org/2020.findings-emnlp.389), [[4]](http://arxiv.org/abs/1909.04925).

So let's re-assign a_results and b_results to just be the seventh item (outputs from the 8th layer) in the tuple containing the hidden states.

In [ ]:
a_results = a_results.hidden_states[7]
b_results = b_results

And I know it seems arbitrary, but let's also get rid of that first dimension in 'em.

In [ ]:
a_results = a_results.squeeze(0) # opposite of unsqueeze
b_results = b_results

We need a way of comparing how close two vectors are--any two vectors. There are a lot of ways to do that. But one really nice way is Cosine Similarity (it's a normalized measurement of distance between two points).

In [ ]:
cos = nn.CosineSimilarity(dim=-1)

And now.... we compare!

In [ ]:
cosine_values = cos(a_results.unsqueeze(1), b_results)

In [ ]:
# what are the dimensions of cosine_values? Why does it have that shape?

I know. We just squeezed the first dimension out these examples. And then we added a MIDDLE dimension to a_results? What the actual hell, man?

Okay. This next bit of code, I don't actually want you to worry about. The whole reason it's here is to help you visualize something. Ignore the code--just run it--and pay attention to the figure it prints out.

In [ ]:
fig = go.Figure(data=go.Heatmap(z=cosine_values.detach().numpy()))
fig.update_xaxes(
    tickmode='array',
    tickvals=list(range(len(b_token_strings))),
    ticktext=b_token_strings
)
fig.update_yaxes(
    tickmode='array',
    tickvals=list(range(len(a_token_strings))),
    ticktext=a_token_strings
)
fig.show()

Cosine similarity is higher when two words have really similar semantic meanings. What do you see in this plot? What does that tell you?

## Convergence-Entropy and Convergence-Information

In [ ]:
import torch.nn as nn
import torch
import numpy as np
import plotly.graph_objects as go
from transformers import AutoTokenizer, AutoModel, AutoConfig

In [ ]:
cos = nn.CosineSimilarity(dim=-1)

In [ ]:
# name of the specific LLM we're going to use/dissect
llm_name = 'roberta-base'

In [ ]:
config = AutoConfig.from_pretrained(llm_name, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(llm_name)
mod = AutoModel.from_pretrained(llm_name, config=config)

Awesome. YOu now have all the requisite knowledge to know how convergence-entropy works. The full equations look like this:

(1) $E_{xi}, E_{yj} = llm(i \in x), llm(j \in y)$

(2) $P(E_{xi}|E_y) = P_{\mathcal{N}_{[0,\infty]}} \bigg( \min_j (CoE(E_{xi}, E_{yj})) \bigg| \mu=0, \sigma \bigg)$

(3) $I(x;y) = - \sum_i \log P(E_{xi}|E_y)$

I know it seems like a lot, but here's the thing: math like this is actually a bit like a recipe. These are all in actuality a series of steps that, if you follow, gets you a particular result. So

### $E_{xi}, E_{yj}$

In [ ]:
a_sentence = 'How many times do I have to tell you: you do not eat the tiny packet that you find in coat pockets in the department store.'
a_token_ids = tokenizer.encode(a_sentence)
a_token_strings = tokenizer.convert_ids_to_tokens(a_token_ids)
a_results = mod(torch.LongTensor(a_token_ids).unsqueeze(0))
a_results = a_results.hidden_states[7]
a_results = a_results.squeeze(0)

In [ ]:
b_sentence = "But they taste like grandma's house!"
b_token_ids = tokenizer.encode(b_sentence)
b_token_strings = tokenizer.convert_ids_to_tokens(b_token_ids)
b_results = mod(torch.LongTensor(b_token_ids).unsqueeze(0))
b_results = b_results.hidden_states[7]
b_results = b_results.squeeze(0)

### $CoE(E_{xi}, E_{yj})$

In [ ]:
CoS = cos(a_results, b_results)
CoE = 1 - CoS

### $\min_j CoE(E_{xi}, E_{yj})$

In [ ]:
CoE = CoE.min(axis=-1) # axis for all the y-data :)

### $P_{N_{[0,\infty]}}(CoE(E_{xi}, E_{yj})|\mu=0, \sigma)$

For reference, scale here is what they call $\sigma$. And what $P_{N_{[0,\infty]}}$ actually means in math speak is a Half-Normal distribution, that starts at 0 and ends in infinity... which is exactly what `torch.distributions.HalfNormal` implements. It's a class that creates a half normal where scale=$\sigma$, it starts at 0, and goes on FOR. EV. ER.

In [ ]:
# Generates unscaled likelihood of minimum CoE data when passed to a
#   half-Gaussian. If you want to scale it, just add log(scale) * nx
#   to outputs.

def half_gaussian_kernel(CoE, sigma=.3):
    return torch.exp(- (CoE**2) / (2*(sigma**2)))

In [ ]:
Pn = half_gaussian_kernel(CoE)

Annoying to do 1 - N.cdf, but it's because the CDF scales in a weird way...

### $-\log P_{N_{[0,\infty]}}(CoE(E_{xi}, E_{yj})|\mu=0, \sigma)$

In [ ]:
logPn = - torch.log(Pn)

In [ ]:
# (1) what is the shape of logPn, and (2) why are some of the information values higher/lower than the others? What do they correlate to, and what does that tell you about where/how information compounds?

In [ ]:
# entropy is the normalized information content, or:
# - pn * torch.log(Pn)

and summing it to get $-\sum \log P_{N_{[0,\infty]}}(CoE(E_{xi}, E_{yj})|\mu=0, \sigma)$

In [ ]:
logPn = logPn.sum()

### Never having to do that again...

In [ ]:
from convergence_entropy.CEDA import ceda_model

mod = ceda_model(
    sigma=.3,
    wv_model=llm_name,
    wv_layers=[7],
    kernel='information'
)

mod.fit(
    x = ['How many times do I have to tell you: you do not eat the tiny packet that you find in coat pockets in the department store.'],
    y= ["But they taste like grandma's house!"]
)

df = mod.graph_df()
df.head()

### Activity!

Go on to linked in or some other social media website. Copy your conversation with someone, turn by turn into a list that looks something like this:

```
x_turns = [turn_1, turn_2, turn_3]
y_turns = [turn_2,turn_3, turn_4]
```

When does information go lower? When does it go higher?

Hint: information is generated when two things are more different from one another :). Think: "Oh, I learned some new information!"